# Seaborn Tutorial for Astronomy Data

This notebook introduces **Seaborn** for visualizing astronomy-style tabular data.

We use a synthetic star-cluster catalogue with Gaia-like quantities:

- RA and Dec
- parallax and distance
- proper motion
- apparent and absolute magnitude
- colour index
- stellar mass
- group labels
- disk classification

The aim is exploratory data analysis: finding patterns before making final publication figures.

## 1. What is Seaborn?

**Seaborn** is a statistical visualization library built on top of Matplotlib.

It is useful in astronomy for:

- colour–magnitude diagrams,
- proper-motion diagrams,
- sky maps,
- histograms and KDE plots,
- box plots and violin plots,
- correlation heatmaps,
- multi-panel plots by cluster or classification.

## 2. Install and import packages

In [ ]:
# Uncomment if needed:
# %pip install -U numpy pandas matplotlib seaborn scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr

sns.set_theme(context="notebook", style="whitegrid", palette="deep")
np.random.seed(42)

## 3. Create a synthetic astronomy catalogue

In [ ]:
def make_group(name, n, ra0, dec0, dist_pc, pmra0, pmdec0, age_myr, disk_fraction):
    ra = np.random.normal(ra0, 0.35, n)
    dec = np.random.normal(dec0, 0.25, n)
    distance = np.random.normal(dist_pc, 12, n)
    parallax = 1000 / distance + np.random.normal(0, 0.08, n)

    pmra = np.random.normal(pmra0, 0.45, n)
    pmdec = np.random.normal(pmdec0, 0.45, n)

    mass = np.random.lognormal(mean=np.log(0.45), sigma=0.55, size=n)
    mass = np.clip(mass, 0.08, 4.0)

    bp_rp = 0.65 + 1.7 * (1 / np.sqrt(mass)) + np.random.normal(0, 0.12, n)
    abs_g = 3.2 + 4.2 * bp_rp - 1.1 * np.log10(mass + 0.05) + 0.25 * age_myr
    gmag = abs_g + 5 * np.log10(distance / 10) + np.random.normal(0, 0.18, n)

    has_disk = np.random.random(n) < disk_fraction
    disk_class = np.where(has_disk, "disk-bearing", "diskless")

    return pd.DataFrame({
        "group": name,
        "ra_deg": ra,
        "dec_deg": dec,
        "distance_pc": distance,
        "parallax_mas": parallax,
        "pmra_masyr": pmra,
        "pmdec_masyr": pmdec,
        "age_myr": age_myr,
        "mass_msun": mass,
        "bp_rp": bp_rp,
        "g_mag": gmag,
        "disk_class": disk_class,
    })

df = pd.concat([
    make_group("Cluster A", 180, 52.2, 31.3, 295, 5.0, -7.8, 1.5, 0.55),
    make_group("Cluster B", 240, 56.1, 32.1, 320, 4.0, -6.2, 3.0, 0.30),
    make_group("Association C", 160, 59.5, 34.2, 340, 2.8, -5.2, 6.0, 0.12),
], ignore_index=True)

df["pm_total_masyr"] = np.sqrt(df["pmra_masyr"]**2 + df["pmdec_masyr"]**2)
df["absolute_g"] = df["g_mag"] - 5 * np.log10(df["distance_pc"] / 10)

display(df.head())
print(df.shape)

## 4. Sky distribution

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="ra_deg", y="dec_deg", hue="group", style="group", s=45, alpha=0.8)
plt.gca().invert_xaxis()
plt.xlabel("Right Ascension [deg]")
plt.ylabel("Declination [deg]")
plt.title("Sky distribution of young stellar groups")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

A real stellar group should cluster not only on the sky, but also in parallax, proper motion, photometry, and youth indicators.

## 5. Proper-motion diagram

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="pmra_masyr", y="pmdec_masyr", hue="group", style="disk_class", s=45, alpha=0.8)
plt.xlabel(r"$\mu_{\alpha*}$ [mas yr$^{-1}$]")
plt.ylabel(r"$\mu_{\delta}$ [mas yr$^{-1}$]")
plt.title("Proper-motion space")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Colour–magnitude diagram

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(data=df, x="bp_rp", y="absolute_g", hue="group", style="disk_class", s=45, alpha=0.8)
plt.gca().invert_yaxis()
plt.xlabel("BP - RP [mag]")
plt.ylabel("Absolute G [mag]")
plt.title("Colour–magnitude diagram")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

Astronomers usually invert magnitude axes because brighter objects have smaller magnitudes.

## 7. Histograms and KDE plots

In [ ]:
plt.figure(figsize=(7, 5))
sns.histplot(data=df, x="distance_pc", hue="group", bins=35, kde=True, element="step", stat="density", common_norm=False)
plt.xlabel("Distance [pc]")
plt.title("Distance distributions")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.kdeplot(data=df, x="mass_msun", hue="group", fill=True, common_norm=False, alpha=0.35)
plt.xlabel("Mass [$M_\odot$]")
plt.title("Smoothed stellar mass distributions")
plt.tight_layout()
plt.show()

KDE plots are useful for quick inspection, but can mislead for small samples, hard boundaries, or incomplete data.

## 8. Box plots and violin plots

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="group", y="pm_total_masyr")
sns.stripplot(data=df, x="group", y="pm_total_masyr", size=2, alpha=0.25)
plt.xlabel("Group")
plt.ylabel("Total proper motion [mas yr$^{-1}$]")
plt.title("Proper-motion spread by group")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.violinplot(data=df, x="group", y="mass_msun", inner="quartile", cut=0)
plt.xlabel("Group")
plt.ylabel("Mass [$M_\odot$]")
plt.title("Mass distribution by group")
plt.tight_layout()
plt.show()

## 9. Pair plots

In [ ]:
small = df[["group", "distance_pc", "pmra_masyr", "pmdec_masyr", "bp_rp", "absolute_g"]].sample(350, random_state=1)

sns.pairplot(
    small,
    hue="group",
    vars=["distance_pc", "pmra_masyr", "pmdec_masyr", "bp_rp", "absolute_g"],
    corner=True,
    plot_kws={"s": 18, "alpha": 0.7}
)
plt.suptitle("Pair plot of selected astronomy parameters", y=1.02)
plt.show()

## 10. Correlation heatmap

In [ ]:
numeric_cols = ["distance_pc", "parallax_mas", "pmra_masyr", "pmdec_masyr", "pm_total_masyr", "mass_msun", "bp_rp", "g_mag", "absolute_g", "age_myr"]
corr = df[numeric_cols].corr(method="spearman")

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", square=True, cmap="vlag", center=0)
plt.title("Spearman rank correlation matrix")
plt.tight_layout()
plt.show()

Correlation does not imply causation. Use heatmaps to generate questions, not final conclusions.

## 11. Regression plot

In [ ]:
plt.figure(figsize=(7, 5))
sns.regplot(data=df, x="bp_rp", y="absolute_g", scatter_kws={"alpha": 0.3, "s": 20}, line_kws={"linewidth": 2})
plt.gca().invert_yaxis()
plt.xlabel("BP - RP [mag]")
plt.ylabel("Absolute G [mag]")
plt.title("Simple regression view of the colour–magnitude trend")
plt.tight_layout()
plt.show()

A straight-line fit is not a stellar-evolution model. For real CMDs, compare with isochrones.

## 12. FacetGrid: one CMD per group

In [ ]:
g = sns.FacetGrid(df, col="group", hue="disk_class", height=4, sharex=True, sharey=True)
g.map_dataframe(sns.scatterplot, x="bp_rp", y="absolute_g", s=25, alpha=0.8)
g.add_legend()
for ax in g.axes.flat:
    ax.invert_yaxis()
    ax.set_xlabel("BP - RP [mag]")
    ax.set_ylabel("Absolute G [mag]")
g.fig.suptitle("Colour–magnitude diagrams by group", y=1.05)
plt.show()

## 13. Disk fraction by group

In [ ]:
disk_fraction = (
    df.assign(has_disk=df["disk_class"].eq("disk-bearing"))
      .groupby("group", as_index=False)
      .agg(disk_fraction=("has_disk", "mean"), n=("has_disk", "size"))
)

display(disk_fraction)

plt.figure(figsize=(7, 5))
sns.barplot(data=disk_fraction, x="group", y="disk_fraction")
plt.ylim(0, 1)
plt.xlabel("Group")
plt.ylabel("Disk fraction")
plt.title("Disk fraction by group")
plt.tight_layout()
plt.show()

Disk fraction is often used as an approximate youth indicator, but it depends on wavelength coverage, sample selection, and classification method.

## 14. Saving figures

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="ra_deg", y="dec_deg", hue="group", s=45, alpha=0.8)
plt.gca().invert_xaxis()
plt.xlabel("Right Ascension [deg]")
plt.ylabel("Declination [deg]")
plt.title("Sky distribution")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("sky_distribution_seaborn.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: sky_distribution_seaborn.png")

# Exercises

## Exercise 1: Parallax distribution
Make a histogram of `parallax_mas` for each group using `hue="group"` and a KDE curve.

## Exercise 2: Disk-bearing stars on the sky
Make a sky plot where points are coloured by `disk_class` and marker style is set by `group`.

## Exercise 3: Proper-motion selection
Select stars with:

```python
4.0 < pmra_masyr < 6.0
-8.5 < pmdec_masyr < -6.5
```

Plot their CMD. Which group dominates?

## Exercise 4: Mass distributions
Use a violin plot or box plot to compare `mass_msun` between groups.

## Exercise 5: Correlation by group
Calculate the Spearman correlation between `bp_rp` and `absolute_g` separately for each group.

## Exercise 6: Publication-style CMD
Create a CMD with:
- inverted y-axis,
- clear labels,
- title,
- legend outside the plot,
- saved as PNG.

## Exercise 7: Challenge
Use `FacetGrid` to make one proper-motion diagram per group, with points coloured by `disk_class`.

## Exercise solution templates

In [ ]:
# Exercise 1 template
# plt.figure(figsize=(7, 5))
# sns.histplot(data=df, x="parallax_mas", hue="group", kde=True, bins=30,
#              element="step", stat="density", common_norm=False)
# plt.xlabel("Parallax [mas]")
# plt.title("Parallax distribution by group")
# plt.show()

In [ ]:
# Exercise 5 template
# for group, sub in df.groupby("group"):
#     rho, pval = spearmanr(sub["bp_rp"], sub["absolute_g"])
#     print(group, "Spearman rho =", round(rho, 3), "p =", round(pval, 3))

# Final notes

Seaborn is excellent for exploratory astronomy plots, especially with Pandas tables.

Good practice:
- label units clearly,
- invert magnitude axes when appropriate,
- state selection criteria,
- avoid over-interpreting visual trends,
- check completeness and selection effects.